# GAS-BayesSHAP — remaining paper experiments (Q1 gap closure)

Gaussian-Adaptive Stratified Bayesian Shapley Estimation (v11.0). This notebook runs the large compute that completes the Q1-readiness review's evidence table: the **N=50 multi-instance sweep** (wine + air), the full matched-budget **curves**, the **width-vs-budget** curve, **Tier-B** group-lag at N=20, and the **R=500 coverage calibration**.

It orchestrates the real CLI entry points (`scripts/run_paper_experiments.py`, `scripts/coverage_validation.py`); it duplicates **no** scientific algorithm. Every output is written with a config-suffixed name (`*_n{N}_budget{B}.csv`) so prior runs are never overwritten, then copied into `main_results/`.

## 1. Environment

In [ ]:
import sys, os, time, json, re, subprocess
from pathlib import Path
import numpy as np
import pandas as pd

sys.path.insert(0, "..")
import gas_bayesshap

ROOT = Path("..").resolve()            # repo root (notebook runs from main_results/)
SCRIPTS = ROOT / "scripts"

print("GAS-BayesSHAP", gas_bayesshap.__version__)
print("repo root  :", ROOT)
wine_csv = ROOT / "data" / "winequality-white.csv"
air_csv  = ROOT / "data" / "Beijing_MultiSite_AirQuality.csv"
print("wine data  :", "OK" if wine_csv.exists() else "MISSING -> loader falls back to synthetic")
print("air  data  :", "OK" if air_csv.exists() else "MISSING -> loader falls back to synthetic")
print("numpy      :", np.__version__, "| pandas:", pd.__version__)

## 2. Configuration

In [ ]:
# Full-size defaults; override every value via env vars for a quick check:
#   GAS_N=2 GAS_BUDGET=200 GAS_TIERB_N=2 GAS_COV_TRIALS=10 GAS_SKIP=1
N          = int(os.environ.get("GAS_N", "50"))          # instances per dataset
EPS        = float(os.environ.get("GAS_EPS", "0.05"))    # tight epsilon
BUDGET     = int(os.environ.get("GAS_BUDGET", "3000"))   # coalition budget
TIERB_N    = int(os.environ.get("GAS_TIERB_N", "20"))    # Tier-B instances
COV_TRIALS = int(os.environ.get("GAS_COV_TRIALS", "500"))# calibration trials
SKIP_HEAVY = os.environ.get("GAS_SKIP", "0") == "1"     # skip curves/widths (smoke mode)

def run(*args, tag=""):
    """Run a CLI script with live output; raise on failure."""
    cmd = [sys.executable, str(SCRIPTS / args[0]), *args[1:]]
    t0 = time.time()
    print(f"\n>>> {tag or ' '.join(args)}")
    r = subprocess.run(cmd, cwd=ROOT)
    dt = time.time() - t0
    if r.returncode != 0:
        raise RuntimeError(f"FAILED ({r.returncode}): {' '.join(args)}")
    print(f"<<< done in {dt/60:.1f} min")
    return dt

print(f"N={N}  eps={EPS}  budget={BUDGET}  tierb_N={TIERB_N}  cov_trials={COV_TRIALS}  skip_heavy={SKIP_HEAVY}")

## 3. RQ1 — wine, N-instance tight-epsilon sweep

In [ ]:
run("run_paper_experiments.py", "--only", "wine", "--n", str(N), "--eps", str(EPS), "--budget", str(BUDGET),
    tag=f"wine  N={N} eps={EPS} budget={BUDGET}")

## 4. RQ2 — air (4 regimes), N-instance tight-epsilon sweep

In [ ]:
run("run_paper_experiments.py", "--only", "air", "--n", str(N), "--eps", str(EPS), "--budget", str(BUDGET),
    tag=f"air   N={N} eps={EPS} budget={BUDGET}")

## 5. RQ2 — matched-budget curves (K = 128 … 2048)

GAS-BayesSHAP vs KernelSHAP vs SamplingSHAP at **identical** coalition budgets. Honest caveat from the committed N=8 curves: KernelSHAP overtakes GAS at K ≥ 1024 on these low-dimensional (M=11) games; GAS wins at low K (≤ 512) where its Bayesian control variate pays off.

In [ ]:
if SKIP_HEAVY:
    print("SKIPPED (GAS_SKIP=1) — full curve sweep is ~1 h on a laptop.")
else:
    run("run_paper_experiments.py", "--only", "curves", tag="matched-budget curves")

## 6. RQ1 — certificate width vs budget

In [ ]:
if SKIP_HEAVY:
    print("SKIPPED (GAS_SKIP=1) — width-vs-budget is ~30 min on a laptop.")
else:
    run("run_paper_experiments.py", "--only", "widths", tag="width-vs-budget")

## 7. RQ3 — Tier-B group-lag game (M = 66 → 11 pollutant macros)

Group-lag Shapley over 11 pollutant macros (6 lags each) for the air-quality task; macro simultaneous coverage is reported. N is no longer capped at 10.

In [ ]:
run("run_paper_experiments.py", "--only", "tierb", "--n", str(TIERB_N),
    tag=f"tierb N={TIERB_N} eps={EPS} budget={BUDGET}")

## 8. Calibration — coverage validation (R repeated trials)

Repeated synthetic trials (M=3 calibration game) to check the anytime empirical-Bernstein certificate: finite-width rate, empirical coverage, coverage-given-finite. The report is parsed from the CLI output and persisted as JSON next to the other paper artifacts. The real-data counterpart is the simultaneous-coverage rate of the N-sweeps above.

In [ ]:
cmd = [sys.executable, str(SCRIPTS / "coverage_validation.py"),
       "--trials", str(COV_TRIALS), "--M", "3", "--epsilon", "1.5",
       "--delta", "0.05", "--max-budget", "300"]
t0 = time.time()
r = subprocess.run(cmd, cwd=ROOT, capture_output=True, text=True)
print(r.stdout)
if r.stderr.strip():
    print(r.stderr)
if r.returncode != 0:
    raise RuntimeError(f"coverage_validation failed ({r.returncode})")

# --- persist the printed report as JSON (same schema as paper_coverage_calibration_R500.json)
KEYS = ("n_trials", "finite_width_rate", "empirical_coverage",
        "coverage_given_finite", "mean_width", "median_width", "max_width",
        "oracle_query_cost (mean)", "oracle_query_cost (max)")
rep = {}
for line in r.stdout.splitlines():
    m = re.match(r"^\s*(.*?)\s*:\s*(.+?)\s*$", line)
    if m and m.group(1) in KEYS:
        key = m.group(1).replace(" ", "_").replace("(", "").replace(")", "")
        val = m.group(2)
        try:
            rep[key] = float(val) if "." in val else int(val)
        except ValueError:
            rep[key] = val
payload = {"n_trials": COV_TRIALS, "M": 3, "epsilon": 1.5, "delta": 0.05,
           "max_budget": 300, **rep}
for dest in (ROOT / "results" / "paper_experiments", ROOT / "main_results"):
    dest.mkdir(parents=True, exist_ok=True)
    (dest / f"paper_coverage_calibration_R{COV_TRIALS}.json").write_text(
        json.dumps(payload, indent=1))
print(f"persisted paper_coverage_calibration_R{COV_TRIALS}.json in "
      f"results/paper_experiments/ and main_results/ ({time.time()-t0:.0f}s)")

## 9. Summary of produced artifacts

In [ ]:
def show(name, path):
    p = Path(path)
    if p.exists():
        print(f"\n[{name}]")
        print(pd.read_csv(p).to_string(index=False))
    else:
        print(f"\n[{name}] NOT FOUND: {p.name}")

show("WINE N-sweep",  ROOT / "main_results" / f"paper_wine_n{N}_budget{BUDGET}_summary.csv")
show("AIR  N-sweep",  ROOT / "main_results" / f"paper_air_n{N}_budget{BUDGET}_summary.csv")
show("TIER-B",        ROOT / "main_results" / "paper_air_tierB_summary.csv")
show("WINE matched-budget", ROOT / "main_results" / "paper_wine_matched_budget.csv")
show("AIR  matched-budget", ROOT / "main_results" / "paper_air_matched_budget.csv")
cov = ROOT / "main_results" / f"paper_coverage_calibration_R{COV_TRIALS}.json"
if cov.exists():
    print(f"\n[COVERAGE CALIBRATION R={COV_TRIALS}]")
    print(json.dumps(json.loads(cov.read_text()), indent=1))

## Notes — expected runtime and honest caveats

- **Measured cost** (sandbox, wine): ≈ 20 ms / coalition-eval ⇒ a budget-3000 instance ≈ 100 s. Full default run ≈ 3.5–4 h on a laptop: wine N=50 ≈ 1.5 h, air N=50 ≈ 1.5 h, curves ≈ 1 h, widths ≈ 30 min, Tier-B N=20 ≈ 30 min, R=500 ≈ 10–20 min.
- **Certificates stay conservative**: past runs give mean certified width ≈ 9 vs attribution scale ≈ 0.3, hence `sign_certified_fraction = 0` and `converged_fraction = 0` (status `BUDGET_EXHAUSTED`) — the intervals are valid but wide.  Width-tightening options: `range_mode="finite_population"` (**rigorous**, Theorem E — the residual marginal is an iid draw from a finite population of C(M-1,s) pairs, so the observed-support range is valid at the realised coverage level 1 − δ2 − δ1 with coupon-collector δ1; R=500 calibration: width 2.38 vs 12.31 spec, coverage 1.0) and `range_mode="empirical_max"` (**heuristic**, flagged `range_bound_is_heuristic=True`).
- **Curves**: KernelSHAP overtakes GAS at K ≥ 1024 on M=11 games; the claim is that GAS wins in the low-budget regime (K ≤ 512), not that it dominates everywhere.
- **Coverage**: the R=500 calibration validates the certificate *mechanism* on a synthetic game; the N-sweep simultaneous-coverage rates carry that check to real data.
- All artifacts land in `results/paper_experiments/` and are copied to `main_results/` with the `paper_` prefix; commit them with the notebook when the run completes.